# Notebook 3. Basket-Horizon Optimization, Monte Carlo, and Investor Output

Purpose:
- define the three Argentine baskets and the three horizon assumptions
- optimize weights inside each basket-horizon cell
- compare methods, run Monte Carlo simulations, and create investor-facing outputs
- publish the analytics layer and presentation visuals


## Imports and Run Settings

This notebook is the investor-facing step of the project. It turns the processed warehouse features into basket-horizon comparisons, simulations, charts, and recommendation tables.


In [1]:
# This cell imports the helpers used for optimization, simulation, uploads, and chart generation.
# Keeping the setup together makes the final notebook easier to explain in a presentation.
from datetime import datetime, timezone

import pandas as pd

from src.bq_utils import get_bigquery_client, query_to_dataframe, upload_dataframe
from src.config import (
    ANALYTICS_TABLES,
    BASKETS,
    DATASETS,
    HORIZONS,
    OPTIMIZATION_METHODS,
    PROCESSED_TABLES,
    PROJECT_ID,
    SIMULATION_SETTINGS,
    basket_rows,
    horizon_rows,
)
from src.portfolio_utils import (
    attach_risk_profiles,
    build_investor_recommendations,
    build_method_comparison,
    build_metric_data_dictionary,
    compute_portfolio_contributions,
    equal_weight_weights,
    evaluate_portfolio,
    optimize_max_sharpe,
    optimize_min_volatility,
    optimize_risk_parity,
)
from src.simulation_utils import run_monte_carlo
from src.visualization_utils import (
    save_basket_horizon_fact_sheet,
    save_investor_profile_overview,
    save_overview_heatmap,
)

RUN_ID = f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
INGESTION_TIMESTAMP = pd.Timestamp.utcnow()
client = get_bigquery_client()
final_value_lookup = {}


## Load Processed Inputs

We read the processed feature layer from BigQuery and build the wide stock return matrix used by the optimizers and simulations.


In [2]:
# This cell loads the processed inputs that feed the final basket-horizon analysis.
# The wide return matrix is the main input for optimization and Monte Carlo work.
asset_returns = query_to_dataframe(
    client,
    f"select * from `{PROJECT_ID}.{DATASETS['processed']}.{PROCESSED_TABLES['asset_returns']}`",
)
stock_metrics = query_to_dataframe(
    client,
    f"select * from `{PROJECT_ID}.{DATASETS['processed']}.{PROCESSED_TABLES['stock_metrics']}`",
)
beta_metrics = query_to_dataframe(
    client,
    f"select * from `{PROJECT_ID}.{DATASETS['processed']}.{PROCESSED_TABLES['beta_metrics']}`",
)

asset_returns['date'] = pd.to_datetime(asset_returns['date'])
returns_wide = asset_returns.pivot_table(
    index='date',
    columns='ticker',
    values='log_return',
    aggfunc='mean',
).sort_index()

print('Processed asset rows:', len(asset_returns))
print('Return matrix shape:', returns_wide.shape)


Processed asset rows: 17800
Return matrix shape: (1780, 10)


## Basket and Horizon Definitions

The basket and horizon definitions are stored explicitly so the warehouse keeps a visible record of the investment universe and the time-risk assumptions.


In [3]:
# This cell materializes the canonical basket and horizon definitions used in the 3x3 comparison matrix.
# We also create simple lookup tables that help enrich the contribution output later.
basket_definitions = pd.DataFrame(basket_rows(RUN_ID, INGESTION_TIMESTAMP))
horizon_definitions = pd.DataFrame(horizon_rows(RUN_ID, INGESTION_TIMESTAMP))

sector_lookup = basket_definitions[['basket_name', 'ticker', 'sector']].drop_duplicates()
stock_type_lookup = stock_metrics[['ticker', 'stock_type']].drop_duplicates() if 'stock_type' in stock_metrics.columns else pd.DataFrame(columns=['ticker', 'stock_type'])

display(basket_definitions.head())
display(horizon_definitions)


,basket_name,basket_label,ticker,basket_order,sector,macro_rationale,run_id,ingestion_timestamp
0,short_term_tactical,Short-Term Tactical Basket,GGAL.BA,1,financials,High beta mix tilted to financials and regulat...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00
1,short_term_tactical,Short-Term Tactical Basket,BMA.BA,2,financials,High beta mix tilted to financials and regulat...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00
2,short_term_tactical,Short-Term Tactical Basket,YPFD.BA,3,energy,High beta mix tilted to financials and regulat...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00
3,short_term_tactical,Short-Term Tactical Basket,EDN.BA,4,utilities,High beta mix tilted to financials and regulat...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00
4,short_term_tactical,Short-Term Tactical Basket,TRAN.BA,5,utilities,High beta mix tilted to financials and regulat...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00


,horizon_name,horizon_label,lookback_days,evaluation_days,simulation_days,max_weight,min_weight,investor_profile_anchor,description,run_id,ingestion_timestamp
0,short_horizon,Short Horizon,126,63,63,0.40,0.05,aggressive,Half-year estimation window with one-quarter e...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00
1,medium_horizon,Medium Horizon,252,126,126,0.35,0.05,balanced,One-year estimation window with half-year eval...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00
2,long_horizon,Long Horizon,504,252,252,0.30,0.05,conservative,Two-year estimation window with one-year evalu...,run_20260426_015224,2026-04-26 01:52:24.114473+00:00


## Optimization, Contributions, and Simulation Loop

This is the core analytical step. For every basket and every horizon, we evaluate four weighting methods, store the weights, measure the historical trade-off, compute stock-level contributions, and run Monte Carlo simulations.


In [4]:
# This cell runs the full 3x3 basket-horizon matrix and stores every method-level result.
# The goal is to compare equal weight, max Sharpe, min volatility, and risk parity in a consistent way.
weight_rows = []
metric_rows = []
contribution_frames = []
simulation_rows = []
path_rows = []

for basket_name, basket_config in BASKETS.items():
    basket_tickers = [ticker for ticker in basket_config['tickers'] if ticker in returns_wide.columns]
    if len(basket_tickers) < 3:
        continue

    basket_returns = returns_wide[basket_tickers].dropna().copy()

    for horizon_name, horizon_config in HORIZONS.items():
        lookback_window = basket_returns.tail(horizon_config['lookback_days']).copy()
        evaluation_window = lookback_window.tail(
            min(len(lookback_window), horizon_config['evaluation_days'])
        ).copy()
        if lookback_window.shape[0] < max(40, len(basket_tickers) * 8):
            continue

        method_weights = {
            'equal_weight': equal_weight_weights(basket_tickers),
            'max_sharpe': optimize_max_sharpe(
                lookback_window,
                min_weight=horizon_config['min_weight'],
                max_weight=horizon_config['max_weight'],
            ),
            'min_volatility': optimize_min_volatility(
                lookback_window,
                min_weight=horizon_config['min_weight'],
                max_weight=horizon_config['max_weight'],
            ),
            'risk_parity': optimize_risk_parity(
                lookback_window,
                min_weight=horizon_config['min_weight'],
                max_weight=horizon_config['max_weight'],
            ),
        }

        macro_risk_note = (
            f"{basket_config['macro_rationale']} "
            'Argentina-specific risks still include regulation, FX pressure, tariff changes, and political sensitivity.'
        )

        for weighting_method in OPTIMIZATION_METHODS:
            weights = method_weights[weighting_method].sort_index()
            basket_horizon_id = f"{basket_name}__{horizon_name}__{weighting_method}"

            metrics = evaluate_portfolio(
                estimation_returns=lookback_window,
                evaluation_returns=evaluation_window,
                weights=weights,
                stock_metrics=stock_metrics,
                beta_metrics=beta_metrics,
            )
            metric_rows.append(
                {
                    'basket_horizon_id': basket_horizon_id,
                    'basket_name': basket_name,
                    'basket_label': basket_config['label'],
                    'horizon_name': horizon_name,
                    'horizon_label': horizon_config['label'],
                    'weighting_method': weighting_method,
                    **metrics,
                    'lookback_days': horizon_config['lookback_days'],
                    'evaluation_days': horizon_config['evaluation_days'],
                    'simulation_days': horizon_config['simulation_days'],
                    'max_weight': horizon_config['max_weight'],
                    'min_weight': horizon_config['min_weight'],
                    'macro_risk_note': macro_risk_note,
                    'ingestion_timestamp': INGESTION_TIMESTAMP,
                    'run_id': RUN_ID,
                }
            )

            ordered_weights = weights.sort_values(ascending=False)
            for weight_rank, (ticker, weight_value) in enumerate(ordered_weights.items(), start=1):
                weight_rows.append(
                    {
                        'basket_horizon_id': basket_horizon_id,
                        'basket_name': basket_name,
                        'horizon_name': horizon_name,
                        'weighting_method': weighting_method,
                        'ticker': ticker,
                        'weight': float(weight_value),
                        'weight_rank': weight_rank,
                        'ingestion_timestamp': INGESTION_TIMESTAMP,
                        'run_id': RUN_ID,
                    }
                )

            contributions = compute_portfolio_contributions(lookback_window, weights)
            contributions['basket_horizon_id'] = basket_horizon_id
            contributions['basket_name'] = basket_name
            contributions['basket_label'] = basket_config['label']
            contributions['horizon_name'] = horizon_name
            contributions['horizon_label'] = horizon_config['label']
            contributions['weighting_method'] = weighting_method
            contributions = contributions.merge(sector_lookup, on=['basket_name', 'ticker'], how='left')
            contributions = contributions.merge(stock_type_lookup, on='ticker', how='left')
            contributions['ingestion_timestamp'] = INGESTION_TIMESTAMP
            contributions['run_id'] = RUN_ID
            contribution_frames.append(contributions)

            monte_carlo_paths, monte_carlo_metrics, final_values = run_monte_carlo(
                returns_window=lookback_window,
                weights=weights,
                simulation_days=horizon_config['simulation_days'],
                num_simulations=SIMULATION_SETTINGS['num_simulations'],
                initial_value=SIMULATION_SETTINGS['initial_value'],
                random_seed=SIMULATION_SETTINGS['random_seed'] + len(simulation_rows),
                store_path_count=SIMULATION_SETTINGS['store_path_count'],
                confidence_level=SIMULATION_SETTINGS['confidence_level'],
            )
            simulation_rows.append(
                {
                    'basket_horizon_id': basket_horizon_id,
                    'basket_name': basket_name,
                    'horizon_name': horizon_name,
                    'weighting_method': weighting_method,
                    **monte_carlo_metrics,
                    'ingestion_timestamp': INGESTION_TIMESTAMP,
                    'run_id': RUN_ID,
                }
            )
            final_value_lookup[basket_horizon_id] = final_values

            monte_carlo_paths['basket_horizon_id'] = basket_horizon_id
            monte_carlo_paths['basket_name'] = basket_name
            monte_carlo_paths['horizon_name'] = horizon_name
            monte_carlo_paths['weighting_method'] = weighting_method
            monte_carlo_paths['ingestion_timestamp'] = INGESTION_TIMESTAMP
            monte_carlo_paths['run_id'] = RUN_ID
            path_rows.append(monte_carlo_paths)

basket_horizon_weights = pd.DataFrame(weight_rows)
basket_horizon_metrics = pd.DataFrame(metric_rows)
basket_horizon_contributions = pd.concat(contribution_frames, ignore_index=True)
monte_carlo_summary = pd.DataFrame(simulation_rows)
monte_carlo_paths = pd.concat(path_rows, ignore_index=True)


## Comparison, Recommendations, and Data Dictionary

After the raw method-level outputs are ready, we classify the risk profile of each basket-horizon allocation, rank the methods, and create the compact data dictionary table.


In [5]:
# This cell turns the method-level outputs into comparison and recommendation tables.
# It also creates the supporting data dictionary table used for academic documentation.
basket_horizon_metrics = attach_risk_profiles(basket_horizon_metrics, monte_carlo_summary)

basket_horizon_method_comparison = build_method_comparison(
    basket_horizon_metrics,
    monte_carlo_summary,
)
basket_horizon_method_comparison['ingestion_timestamp'] = INGESTION_TIMESTAMP
basket_horizon_method_comparison['run_id'] = RUN_ID

investor_recommendation_summary = build_investor_recommendations(
    basket_horizon_method_comparison
)
investor_recommendation_summary['ingestion_timestamp'] = INGESTION_TIMESTAMP
investor_recommendation_summary['run_id'] = RUN_ID

metric_data_dictionary = build_metric_data_dictionary(RUN_ID, INGESTION_TIMESTAMP)

display(
    basket_horizon_method_comparison.sort_values(
        ['horizon_name', 'basket_rank_within_horizon', 'method_rank_within_basket_horizon']
    ).head(12)
)
display(investor_recommendation_summary)


,basket_horizon_id,basket_name,basket_label,horizon_name,horizon_label,weighting_method,expected_portfolio_return,portfolio_volatility,weighted_sharpe,sortino_ratio,...,equal_weight_probability_of_loss,equal_weight_return_delta,equal_weight_sharpe_delta,equal_weight_probability_of_loss_delta,method_rank_within_basket_horizon,is_best_method_for_basket_horizon,basket_best_selection_score,basket_rank_within_horizon,is_best_basket_for_horizon,horizon_winner_reason
9,short_term_tactical__long_horizon__max_sharpe,short_term_tactical,Short-Term Tactical Basket,long_horizon,Long Horizon,max_sharpe,0.477831,0.475171,1.005597,1.450692,...,0.1820,0.046109,0.131397,-0.0305,1.0,True,3.949977,1.0,True,Highest upside after applying no-short and max...
10,short_term_tactical__long_horizon__min_volatility,short_term_tactical,Short-Term Tactical Basket,long_horizon,Long Horizon,min_volatility,0.467864,0.474396,0.986230,1.317803,...,0.1820,0.036142,0.112030,-0.0195,2.0,False,3.949977,1.0,False,Highest upside after applying no-short and max...
11,short_term_tactical__long_horizon__risk_parity,short_term_tactical,Short-Term Tactical Basket,long_horizon,Long Horizon,risk_parity,0.440531,0.485772,0.906868,1.136009,...,0.1820,0.008809,0.032668,0.0105,3.0,False,3.949977,1.0,False,Highest upside after applying no-short and max...
8,short_term_tactical__long_horizon__equal_weight,short_term_tactical,Short-Term Tactical Basket,long_horizon,Long Horizon,equal_weight,0.431722,0.493848,0.874200,1.050528,...,0.1820,0.000000,0.000000,0.0000,4.0,False,3.949977,1.0,False,Highest upside after applying no-short and max...
33,long_term_structural__long_horizon__max_sharpe,long_term_structural,Long-Term Structural Basket,long_horizon,Long Horizon,max_sharpe,0.410426,0.418966,0.979615,1.108861,...,0.2095,0.092539,0.191744,-0.0410,1.0,True,3.651497,2.0,False,Highest upside after applying no-short and max...
32,long_term_structural__long_horizon__equal_weight,long_term_structural,Long-Term Structural Basket,long_horizon,Long Horizon,equal_weight,0.317886,0.403475,0.787871,1.094729,...,0.2095,0.000000,0.000000,0.0000,2.0,False,3.651497,2.0,False,Best diversification-adjusted trade-off for th...
35,long_term_structural__long_horizon__risk_parity,long_term_structural,Long-Term Structural Basket,long_horizon,Long Horizon,risk_parity,0.297514,0.396286,0.750756,1.027164,...,0.2095,-0.020372,-0.037115,0.0045,3.0,False,3.651497,2.0,False,Best diversification-adjusted trade-off for th...
34,long_term_structural__long_horizon__min_volati...,long_term_structural,Long-Term Structural Basket,long_horizon,Long Horizon,min_volatility,0.271117,0.380881,0.711816,0.840026,...,0.2095,-0.046769,-0.076055,0.0280,4.0,False,3.651497,2.0,False,Best diversification-adjusted trade-off for th...
21,medium_term_balanced__long_horizon__max_sharpe,medium_term_balanced,Medium-Term Balanced Basket,long_horizon,Long Horizon,max_sharpe,0.382605,0.421414,0.907908,1.292219,...,0.2485,0.084008,0.185575,-0.0645,1.0,True,3.546574,3.0,False,Highest upside after applying no-short and max...
20,medium_term_balanced__long_horizon__equal_weight,medium_term_balanced,Medium-Term Balanced Basket,long_horizon,Long Horizon,equal_weight,0.298597,0.413378,0.722334,0.908466,...,0.2485,0.000000,0.000000,0.0000,2.0,False,3.546574,3.0,False,Best diversification-adjusted trade-off for th...


,recommendation_scope,recommendation_key,investor_profile,horizon_name,basket_name,weighting_method,basket_horizon_id,recommendation_rank,selection_score,recommendation_reason,key_risk,ingestion_timestamp,run_id
0,horizon,short_horizon,None,short_horizon,short_term_tactical,max_sharpe,short_term_tactical__short_horizon__max_sharpe,1,9.370071,Best downside resilience after horizon-specifi...,High concentration risk remains inside the bas...,2026-04-26 01:52:24.114473+00:00,run_20260426_015224
1,horizon,medium_horizon,None,medium_horizon,short_term_tactical,max_sharpe,short_term_tactical__medium_horizon__max_sharpe,1,7.723362,Highest upside after applying no-short and max...,High concentration risk remains inside the bas...,2026-04-26 01:52:24.114473+00:00,run_20260426_015224
2,horizon,long_horizon,None,long_horizon,short_term_tactical,max_sharpe,short_term_tactical__long_horizon__max_sharpe,1,3.949977,Highest upside after applying no-short and max...,High concentration risk remains inside the bas...,2026-04-26 01:52:24.114473+00:00,run_20260426_015224
3,investor_profile,conservative,conservative,short_horizon,short_term_tactical,max_sharpe,short_term_tactical__short_horizon__max_sharpe,1,9.370071,Lowest downside probability after comparing al...,High concentration risk remains inside the bas...,2026-04-26 01:52:24.114473+00:00,run_20260426_015224
4,investor_profile,balanced,balanced,short_horizon,short_term_tactical,max_sharpe,short_term_tactical__short_horizon__max_sharpe,1,9.370071,"Best overall mix of Sharpe, diversification, a...",High concentration risk remains inside the bas...,2026-04-26 01:52:24.114473+00:00,run_20260426_015224
5,investor_profile,aggressive,aggressive,short_horizon,short_term_tactical,max_sharpe,short_term_tactical__short_horizon__max_sharpe,1,9.370071,Highest upside expectation after optimization ...,High concentration risk remains inside the bas...,2026-04-26 01:52:24.114473+00:00,run_20260426_015224


## Presentation Visuals

The chart outputs are created here so they can be dropped directly into the final presentation. Each fact sheet uses the winning method for one basket-horizon cell.


In [6]:
# This cell creates the two overview charts and the nine basket-horizon fact sheets.
# The fact sheets always use the best method inside each basket-horizon cell.
chart_paths = []
chart_paths.append(save_overview_heatmap(basket_horizon_method_comparison))
chart_paths.append(
    save_investor_profile_overview(
        investor_recommendation_summary,
        basket_horizon_method_comparison,
    )
)

winners = basket_horizon_method_comparison[
    basket_horizon_method_comparison['is_best_method_for_basket_horizon']
].copy()
monte_carlo_lookup = monte_carlo_summary.set_index('basket_horizon_id')

for _, winner_row in winners.iterrows():
    basket_horizon_id = winner_row['basket_horizon_id']
    weight_slice = basket_horizon_weights[
        basket_horizon_weights['basket_horizon_id'] == basket_horizon_id
    ].copy()
    contribution_slice = basket_horizon_contributions[
        basket_horizon_contributions['basket_horizon_id'] == basket_horizon_id
    ].copy()
    simulation_row = monte_carlo_lookup.loc[basket_horizon_id]
    final_values = final_value_lookup[basket_horizon_id]

    chart_paths.append(
        save_basket_horizon_fact_sheet(
            weight_slice,
            contribution_slice,
            winner_row,
            simulation_row,
            final_values,
        )
    )

print('Charts written:')
for chart_path in chart_paths:
    print(f'- {chart_path}')


Charts written:
- outputs/charts/overview_basket_horizon_heatmap.png
- outputs/charts/overview_investor_profile_recommendations.png
- outputs/charts/short_term_tactical__short_horizon__fact_sheet.png
- outputs/charts/short_term_tactical__medium_horizon__fact_sheet.png
- outputs/charts/short_term_tactical__long_horizon__fact_sheet.png
- outputs/charts/medium_term_balanced__short_horizon__fact_sheet.png
- outputs/charts/medium_term_balanced__medium_horizon__fact_sheet.png
- outputs/charts/medium_term_balanced__long_horizon__fact_sheet.png
- outputs/charts/long_term_structural__short_horizon__fact_sheet.png
- outputs/charts/long_term_structural__medium_horizon__fact_sheet.png
- outputs/charts/long_term_structural__long_horizon__fact_sheet.png


## Upload Analytics Tables

We now publish the analytics layer. Static definition tables use full refresh, while run-based analytics use append mode with `run_id` cleanup through the upload helper.


In [9]:
# This cell writes the notebook outputs back to BigQuery.
# Static definition tables use truncate, while run-level analytics keep append history by run_id.
upload_dataframe(
    client,
    basket_definitions,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['basket_definitions'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    horizon_definitions,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['horizon_definitions'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    basket_horizon_weights,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['basket_horizon_weights'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    basket_horizon_metrics,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['basket_horizon_metrics'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    basket_horizon_method_comparison,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['basket_horizon_method_comparison'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    basket_horizon_contributions,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['basket_horizon_contributions'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    monte_carlo_paths,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['monte_carlo_paths'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    monte_carlo_summary,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['monte_carlo_summary'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    investor_recommendation_summary,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['investor_recommendation_summary'],
    'WRITE_TRUNCATE',
    run_id=RUN_ID,
)
upload_dataframe(
    client,
    metric_data_dictionary,
    PROJECT_ID,
    DATASETS['analytics'],
    ANALYTICS_TABLES['metric_data_dictionary'],
    'WRITE_TRUNCATE',
)

print('Notebook 3 tables written:')
for table_name in [
    'basket_definitions',
    'horizon_definitions',
    'basket_horizon_weights',
    'basket_horizon_metrics',
    'basket_horizon_method_comparison',
    'basket_horizon_contributions',
    'monte_carlo_paths',
    'monte_carlo_summary',
    'investor_recommendation_summary',
    'metric_data_dictionary',
]:
    print(f"- {PROJECT_ID}.{DATASETS['analytics']}.{ANALYTICS_TABLES[table_name]}")


Notebook 3 tables written:
- bigdata-financeargentina.analytics_market.basket_definitions
- bigdata-financeargentina.analytics_market.horizon_definitions
- bigdata-financeargentina.analytics_market.basket_horizon_weights
- bigdata-financeargentina.analytics_market.basket_horizon_metrics
- bigdata-financeargentina.analytics_market.basket_horizon_method_comparison
- bigdata-financeargentina.analytics_market.basket_horizon_contributions
- bigdata-financeargentina.analytics_market.monte_carlo_paths
- bigdata-financeargentina.analytics_market.monte_carlo_summary
- bigdata-financeargentina.analytics_market.investor_recommendation_summary
- bigdata-financeargentina.analytics_market.metric_data_dictionary


## Final Summary

The final cell prints the horizon winners so a reviewer can quickly see which basket-method combination won at each investment horizon.


In [10]:
# This cell prints a short operational summary for the final project outputs.
# It helps a reviewer see the main winners without searching across multiple tables.
horizon_winners = investor_recommendation_summary[
    investor_recommendation_summary['recommendation_scope'] == 'horizon'
].sort_values('horizon_name')

print('=' * 80)
print('Argentina Portfolio Risk Intelligence Pipeline - Basket-Horizon Summary')
print('=' * 80)
for _, row in horizon_winners.iterrows():
    print(
        f"{row['horizon_name']}: {row['basket_name']} | {row['weighting_method']} | score={row['selection_score']:.2f}"
    )
print('=' * 80)


Argentina Portfolio Risk Intelligence Pipeline - Basket-Horizon Summary
long_horizon: short_term_tactical | max_sharpe | score=3.95
medium_horizon: short_term_tactical | max_sharpe | score=7.72
short_horizon: short_term_tactical | max_sharpe | score=9.37
